# Clasificación de Dígitos MNIST - Versión Compatible

Versión actualizada para generar modelo compatible con versiones modernas de TensorFlow

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
import math
import logging

logger = tf.get_logger()
logger.setLevel(logging.ERROR)

print('TensorFlow versión:', tf.__version__)

TensorFlow versión: 2.19.0


In [2]:
print('Cargando el dataset MNIST...')

(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

print('Dataset cargado exitosamente')

Cargando el dataset MNIST...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/mnist/incomplete.IG1ULP_3.0.1/mnist-train.tfrecord*...:   0%|          | 0…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/mnist/incomplete.IG1ULP_3.0.1/mnist-test.tfrecord*...:   0%|          | 0/…

Dataset mnist downloaded and prepared to /root/tensorflow_datasets/mnist/3.0.1. Subsequent calls will reuse this data.
Dataset cargado exitosamente


In [3]:
def normalize(images, labels):
    images = tf.cast(images, tf.float32)
    images /= 255
    return images, labels

train_dataset = ds_train.map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = ds_test.map(normalize, num_parallel_calls=tf.data.AUTOTUNE)

print('Datos normalizados')

Datos normalizados


In [4]:
print('Definiendo la estructura de la CNN...')

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),

    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
])

print('Modelo CNN definido')
model.summary()


Definiendo la estructura de la CNN...
Modelo CNN definido


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 121,930 (476.29 KB)

 Trainable params: 121,930 (476.29 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('Modelo compilado')

Modelo compilado


In [6]:
BATCHSIZE = 32

train_dataset = train_dataset.cache()
train_dataset = train_dataset.shuffle(ds_info.splits['train'].num_examples)
train_dataset = train_dataset.batch(BATCHSIZE)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

test_dataset = test_dataset.batch(BATCHSIZE)
test_dataset = test_dataset.cache()
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

print('Datos preparados para entrenamiento')

Datos preparados para entrenamiento


In [7]:
print('Comenzando el entrenamiento del modelo (5 épocas)...')

history = model.fit(
    train_dataset,
    epochs=5,
    validation_data=test_dataset,
    verbose=1
)

print('Entrenamiento completado')

Comenzando el entrenamiento del modelo (5 épocas)...
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 64s 30ms/step - accuracy: 0.8801 - loss: 0.3861 - val_accuracy: 0.9803 - val_loss: 0.0610
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 54s 29ms/step - accuracy: 0.9771 - loss: 0.0751 - val_accuracy: 0.9885 - val_loss: 0.0344
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 52s 28ms/step - accuracy: 0.9850 - loss: 0.0481 - val_accuracy: 0.9885 - val_loss: 0.0321
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 51s 27ms/step - accuracy: 0.9889 - loss: 0.0358 - val_accuracy: 0.9905 - val_loss: 0.0292
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 50s 27ms/step - accuracy: 0.9896 - loss: 0.0325 - val_accuracy: 0.9896 - val_loss: 0.0345
Entrenamiento completado


In [8]:
print('Evaluando el modelo...')

test_loss, test_accuracy = model.evaluate(
    test_dataset,
    verbose=1
)

print(f'Precisión en datos de prueba: {test_accuracy:.4f}')

Evaluando el modelo...
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9897 - loss: 0.0350
Precisión en datos de prueba: 0.9896


In [9]:
print('Guardando modelo compatible...')

model.save('modelo_MINST_completo.keras')

print('Modelo guardado como modelo_MINST_completo.keras')
print('Modelo compatible generado exitosamente!')

Guardando modelo compatible...
Modelo guardado como modelo_MINST_completo.keras
Modelo compatible generado exitosamente!


In [10]:
print('Probando el modelo guardado...')

loaded_model = tf.keras.models.load_model('modelo_MINST_completo.keras')

loaded_test_loss, loaded_test_accuracy = loaded_model.evaluate(
    test_dataset,
    verbose=1
)

print(f'Precisión del modelo cargado: {loaded_test_accuracy:.4f}')
print('Modelo guardado y cargado correctamente!')

Probando el modelo guardado...
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9897 - loss: 0.0350
Precisión del modelo cargado: 0.9896
Modelo guardado y cargado correctamente!
